In [ ]:
import torch
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# Your data prep (from previous)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)

# SMOTE (optional but recommended)
smote = SMOTE(random_state=seed)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# After SMOTE...
X_train_np = X_train_smote.values
y_train_np = y_train_smote.values.ravel()
X_test_np = X_test.values
y_test_np = y_test.values.ravel()

# TabNet - MAX RECALL TUNING
tabnet = TabNetClassifier(
    n_d=64, n_a=64,       # attention dimensions
    n_steps=5,            # decision steps  
    gamma=1.5,            # sparsity coefficient
    lambda_sparse=1e-4,   # feature selection strength
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=0.02),
    mask_type="entmax",   # sparse attention
    scheduler_params={"step_size": 50, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    verbose=1
)

# Train
tabnet.fit(
    X_train_np, y_train_np,
    eval_set=[(X_test_np, y_test_np)],
    eval_metric=['logloss'],
    max_epochs=200,
    patience=20,          # early stopping
    batch_size=1024,
    virtual_batch_size=128
)

# Predict & evaluate
y_pred_tabnet = tabnet.predict(X_test_np)
y_proba_tabnet = tabnet.predict_proba(X_test_np)[:, 1]

print("=== TABNET RESULTS ===")
print(classification_report(y_test_np, y_pred_tabnet, digits=3))
print(f"ROC AUC: {roc_auc_score(y_test_np, y_proba_tabnet):.3f}")

# Feature importance (built-in!)
importances = tabnet.feature_importances_
print("\nTop 5 features:")
for i, (name, imp) in enumerate(zip(X.columns, importances)):
    if i < 5:
        print(f"{name}: {imp:.3f}")

In [ ]:
#path = os.getenv("TEST_PATH")
path = r"..\Data\SyntheticData\synthetic_data_2026_03_25_04_49_13.csv"
df_test = pd.read_csv(path)
target = "Problem_SKU"
seed = 1337 
# One-hot encode Storage_Size (drop size_4 as baseline)
size_dummies_test = pd.get_dummies(df_test['Storage_Size'], prefix='Size', drop_first=True).astype(int)

# Encode Defect_In_Linked_Receive as 0/1
defect_linked_num_test = df_test['Defect_In_Linked_Receive'].astype(int)

# Numeric features (keep standardized)
numeric_features = [
    "Global_SKU_Defect_Rate_%_std",
    "ABS_Volume_Difference_std",
    "Aisle_Hold_%_std",
    "#_Pick_Events_std",
    "#_Pick_Events_In_Clique_std",
    "#_Picks_std",
    "#_Picks_In_Clique_std",
    "Time_In_Loc_std",
    "Current_Max_Volume_std",
]

feature_cols_test = numeric_features + list(size_dummies_test.columns) + ['Defect_In_Linked_Receive']

# Combine all properly encoded features
X_test = df_test[numeric_features].copy()
X_test = pd.concat([X_test, size_dummies_test, defect_linked_num_test], axis=1)
y_test = df_test[target]

X_train_test, X_test_test, y_train_test, y_test_test = train_test_split(
    X_test, y_test, test_size=0.95, random_state=seed, stratify=y_test   
)

# Predict & evaluate
y_pred_tabnet = tabnet.predict(X_test_test.values)
y_proba_tabnet = tabnet.predict_proba(X_test_test.values)[:, 1]

print("=== TABNET RESULTS ===")
print(classification_report(y_test_test.values, y_pred_tabnet, digits=3))
print(f"ROC AUC: {roc_auc_score(y_test_test.values, y_proba_tabnet):.3f}")

# Feature importance (built-in!)
importances = tabnet.feature_importances_
print("\nTop 5 features:")
for i, (name, imp) in enumerate(zip(X.columns, importances)):
    if i < 5:
        print(f"{name}: {imp:.3f}")